# Which Content Should Editors Refresh First?



This capstone mirrors the deployed research paper. It assembles the reproducible receipts from the FlyRank starter pipeline, grouped model comparison, and action playbook. The output is a human-reviewed refresh queue, not an automatic publishing system.

In [1]:
from pathlib import Path

import json



import pandas as pd



candidates = [Path.cwd(), *Path.cwd().parents]

root = next(

    (candidate for candidate in candidates if (candidate / "outputs" / "model_results.json").exists()),

    None,

)

assert root is not None, "Run this notebook from inside the repository after python scripts/run_all.py."



model_results = json.loads((root / "outputs" / "model_results.json").read_text())

summary = json.loads((root / "outputs" / "summary.json").read_text())

playbook = json.loads((root / "work" / "outputs" / "action_playbook.json").read_text())

print(f"Loaded receipts for {model_results['input_rows']:,} scored pages.")

Loaded receipts for 30,000 scored pages.


## 1. Question



Which existing pages should an editor inspect first when refresh capacity is limited? The unit is one pseudonymized content item, and the output is a ranked score with an action and reason codes. False positives consume review time; false negatives may leave declining, still-visible pages unattended.

In [2]:
question_receipt = {

    "unit": "pseudonymized content item",

    "output": "ranked refresh-review queue",

    "decision_owner": "human editor or strategist",

    "primary_metric": "precision_at_50",

}

pd.Series(question_receipt)

unit               pseudonymized content item
output            ranked refresh-review queue
decision_owner     human editor or strategist
primary_metric                precision_at_50
dtype: object

## 2. Data



The bundled FlyRank anonymized starter release contains 30,000 content items from 32 pseudonymous clients and 44 columns. Metrics describe a trailing 90-day snapshot; exact calendar bounds are not exposed, so this work makes no time-forward claim. Rate fields are percentages multiplied by 100, and `avg_position = 0` means missing position data. No names, URLs, domains, titles, keywords, or raw queries are used or displayed.

In [3]:
data_receipt = pd.Series(

    {

        "release": "bundled anonymized starter release",

        "rows": model_results["input_rows"],

        "clients": 32,

        "grain": "one row per pseudonymized content item",

        "window": "trailing 90-day snapshot; calendar bounds unavailable",

        "observed_decline_rate": model_results["target_positive_rate"],

    }

)

data_receipt

release                                 bundled anonymized starter release
rows                                                                 30000
clients                                                                 32
grain                               one row per pseudonymized content item
window                   trailing 90-day snapshot; calendar bounds unav...
observed_decline_rate                                             0.542067
dtype: object

## 3. Methodology



The target is `is_declining_label = (trend_direction == "down")`, an observed in-window proxy. `trend_direction`, `trend_pct`, overlapping recent-window fields, provider/model metadata, and identifiers are excluded from features. The baseline is a transparent stale/visible, low-CTR, thin-content, and page-one-age rule. Logistic Regression, a constrained Decision Tree, and a constrained Random Forest are compared on the same complete-client holdout with seed 42. Model selection prioritizes precision@50.

In [4]:
method_receipt = pd.Series(

    {

        "target": model_results["target"],

        "split": model_results["split_strategy"],

        "train_rows": model_results["train_rows"],

        "test_rows": model_results["test_rows"],

        "seed": 42,

        "feature_count_after_encoding": model_results["feature_count"],

        "selected_model": model_results["best_model"]["name"],

    }

)

method_receipt

target                          is_declining_label
split                               client_holdout
train_rows                                   27675
test_rows                                     2325
seed                                            42
feature_count_after_encoding                    52
selected_model                       random_forest
dtype: object

## 4. Results (vs baseline)



On the reference client holdout, the Random Forest places 37 observed declining-label pages in the first 50 recommendations, compared with 12 for the rule baseline. Precision@50 rises from 0.24 to 0.74 and average precision rises from 0.468 to 0.618. These are measured ranking results on held-out clients, not estimates of refresh impact.

In [5]:
comparison_rows = []

baseline = model_results["baseline"]

comparison_rows.append(

    {

        "method": "rule_baseline",

        "roc_auc": baseline["baseline_roc_auc"],

        "average_precision": baseline["baseline_average_precision"],

        "precision_at_20": baseline["baseline_precision_at_20"],

        "precision_at_50": baseline["baseline_precision_at_50"],

        "precision_at_100": baseline["baseline_precision_at_100"],

    }

)

for name, metrics in model_results["models"].items():

    comparison_rows.append(

        {

            "method": name,

            "roc_auc": metrics["roc_auc"],

            "average_precision": metrics["average_precision"],

            "precision_at_20": metrics["precision_at_20"],

            "precision_at_50": metrics["precision_at_50"],

            "precision_at_100": metrics["precision_at_100"],

        }

    )

comparison = pd.DataFrame(comparison_rows).set_index("method")

comparison.round(3)

,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100
method,,,,,
rule_baseline,0.627,0.468,0.15,0.24,0.36
decision_tree,0.742,0.575,0.50,0.54,0.53
logistic_regression,0.700,0.522,0.35,0.40,0.44
random_forest,0.750,0.618,0.65,0.74,0.72


## 5. Limitations



The label describes decline inside the same snapshot rather than a future outcome. The client holdout is not a time split, only 32 clients are available, and model selection changes across grouped holdouts: the separate Week-5 split selected Logistic Regression at precision@50 = 0.72 versus a 0.38 baseline. Importance is associational, and no randomized refresh experiment has measured incremental traffic. Use the result for human-reviewed decision support only.

In [6]:
limitations = [

    "in-window proxy target, not a future label",

    "client holdout, not time-forward validation",

    "32 clients and split-sensitive model selection",

    "importance is model reliance, not causality",

    "no randomized estimate of refresh impact",

    "human review required before action",

]

pd.DataFrame({"limitation": limitations})

,limitation
0,"in-window proxy target, not a future label"
1,"client holdout, not time-forward validation"
2,32 clients and split-sensitive model selection
3,"importance is model reliance, not causality"
4,no randomized estimate of refresh impact
5,human review required before action


## 6. Ranked recommendations



Begin with high-confidence visible pages and inspect the reasons. Review CTR and search intent first when visibility is strong but clicks are weak; review engagement when sessions are meaningful but engagement signals are weak; expand only after confirming a coverage gap; monitor low-confidence items. Never automate publication, deletion, redirects, outreach, or promises of traffic recovery.

In [7]:
recommendations = pd.DataFrame(playbook["action_priority"]).sort_values("priority")

recommendations

,priority,action,human_next_step
0,1,refresh_and_review_ctr,"Check intent, title/snippet alignment, and liv..."
1,2,refresh_and_review_engagement,Check whether the page satisfies intent and wh...
2,3,expand_and_refresh,Confirm a real coverage gap before adding cont...
3,4,refresh,"Review freshness, accuracy, and business prior..."
4,5,monitor,Defer unless new evidence raises priority.


## 7. Artifacts the paper embeds



The deployed paper uses the model comparison receipt plus the generated feature-importance and action-mix charts. The full row-level queue remains a generated artifact; the public paper presents aggregate counts and an identifier-free recommendation preview.

In [8]:
artifact_paths = [

    root / "outputs" / "model_results.json",

    root / "outputs" / "summary.json",

    root / "outputs" / "model_report.md",

    root / "outputs" / "charts" / "top_feature_importance.svg",

    root / "outputs" / "charts" / "action_mix.svg",

    root / "work" / "outputs" / "action_playbook.json",

    root / "work" / "capstone_report.md",

]

artifact_check = pd.DataFrame(

    {"artifact": [str(path.relative_to(root)) for path in artifact_paths], "exists": [path.exists() for path in artifact_paths]}

)

assert artifact_check["exists"].all(), "Run the pipeline and action playbook before publishing."

artifact_check

,artifact,exists
0,outputs/model_results.json,True
1,outputs/summary.json,True
2,outputs/model_report.md,True
3,outputs/charts/top_feature_importance.svg,True
4,outputs/charts/action_mix.svg,True
5,work/outputs/action_playbook.json,True
6,work/capstone_report.md,True


## Reproducibility and credit



Run `pip install -r requirements.txt` and then `python scripts/run_all.py` from a fresh clone. The pipeline uses seed 42 and writes its receipts to `outputs/`. Supporting notebooks and the paper live under `work/`.



Built on the [FlyRank ML Internship dataset](https://flyrank.ai/).



## Self-check



- [x] Every section contains reasoning and supporting code

- [x] All code cells executed in order with no errors

- [x] No client names, URLs, private queries, or row identifiers are displayed

- [x] Claims use observed, measured, directional, and decision-support language

- [ ] Deploy the paper and record its direct URL in `submission/paper_url.txt`